# Exercise 2. Annotate and Classify on Few Examples
So far we have been working with pre-labeled datasets. However, many interesting NLP questions are also to be found in datasets that do not have any labels!

**Let's imagine that we want to expand our student questions dataset** with *history* and *social science* questions:
> Can a country have elections but not be democratic? Argue for your answer.

> Who led the civil rights movement in the United States? 
> A. Nelson Mandela
> B. Martin Luther King Jr.
> C. Malcolm X
> D. Rosa Parks

## 1.1 Let's Annotate!
:::{admonition} HANDS-ON
:class: red
Below this box, there will be 16 questions that you need to annotate as either "Social Science" or "History". Read through them carefully and annotate by yourself. Then compare with a friend. Consider these: 
- Do you agree on the annotations? 
- Were there any difficult ones?

I have tried to design the questions so that there is 8 per class, but you might disagree here!
:::


Copy the dictionary of questions into your own notebook/script and replace "LABEL" with `"Social Science"` or `"History"`:

In [35]:
questions = {
    "Societies collapse when their political institutions fail. Do you agree?": "LABEL",
    "Can a country have elections but not be democratic? Argue for your answer.": "LABEL",
    "Who led the civil rights movement in the United States? \nA. Nelson Mandela\nB. Martin Luther King Jr.\nC. Malcolm X\nD. Rosa Parks": "LABEL",
    "When empires expand, is it always for economic reasons? Discuss with examples.": "LABEL",
    "Which organization was formed after World War II to promote peace and cooperation?\nA. League of Nations\nB. NATO\nC. United Nations\nD. European Union": "LABEL",
    "What is the main difference between capitalism and socialism?": "LABEL",
    "Nationalism can emerge from social institutions or historical events. Which factor do you think has been more influential in shaping modern states?": "LABEL",
    "Social progress does not always follow technological progress. Provide one example where technology increased inequality rather than reducing it.": "LABEL",
    "Which factor most contributed to rapid urban growth in 19th-century Europe?\nA. Agricultural decline\nB. Industrial job opportunities\nC. Rise of universities\nD. Religious reforms": "LABEL",
    "Social movements can succeed through protests, legal change, or public opinion. Which has been most effective? Explain with one example.": "LABEL",
    "How did the introduction of cash crops in colonial Africa reshape local social structures?\nA. Increased wealth for local farmers\nB. Strengthened traditional hierarchies\nC. Created labor inequalities\nD. All of the above": "LABEL",
    "Which factor contributed most to the spread of Islam across Africa and Asia before 1500 CE?\nA. Trade networks\nB. Military conquest\nC. Cultural assimilation\nD. Missionary activity": "LABEL",
    "Which technological innovation in the Indian Ocean trade had the largest societal effect before 1600 CE?\nA. Lateen sails\nB. Compass navigation\nC. Shipbuilding techniques\nD. Port infrastructure": "LABEL",
    "Which has a stronger influence on social behavior: historical narratives or contemporary media? Explain briefly.": "LABEL",
    "Which is more effective in promoting public health: top-down government interventions or grassroots social initiatives? Explain.": "LABEL",
    "Which factor most influences political participation in modern societies?\nA. Economic stability\nB. Education\nC. Media exposure\nD. Social networks": "LABEL"
}

In [36]:
# MY ANNOTATIONS
questions = {
    "Societies collapse when their political institutions fail. Do you agree?": "Social Science",
    "Can a country have elections but not be democratic? Argue for your answer.": "Social Science",
    "Who led the civil rights movement in the United States? \nA. Nelson Mandela\nB. Martin Luther King Jr.\nC. Malcolm X\nD. Rosa Parks": "History",
    "When empires expand, is it always for economic reasons? Discuss with examples.": "History",
    "Which organization was formed after World War II to promote peace and cooperation?\nA. League of Nations\nB. NATO\nC. United Nations\nD. European Union": "History",
    "What is the main difference between capitalism and socialism?": "Social Science",
    "Nationalism can emerge from social institutions or historical events. Which factor do you think has been more influential in shaping modern states?": "History",
    "Social progress does not always follow technological progress. Provide one example where technology increased inequality rather than reducing it.": "Social Science",
    "Which factor most contributed to rapid urban growth in 19th-century Europe?\nA. Agricultural decline\nB. Industrial job opportunities\nC. Rise of universities\nD. Religious reforms": "History",
    "Social movements can succeed through protests, legal change, or public opinion. Which has been most effective? Explain with one example.": "Social Science",
    "How did the introduction of cash crops in colonial Africa reshape local social structures?\nA. Increased wealth for local farmers\nB. Strengthened traditional hierarchies\nC. Created labor inequalities\nD. All of the above": "History",
    "Which factor contributed most to the spread of Islam across Africa and Asia before 1500 CE?\nA. Trade networks\nB. Military conquest\nC. Cultural assimilation\nD. Missionary activity": "History",
    "Which technological innovation in the Indian Ocean trade had the largest societal effect before 1600 CE?\nA. Lateen sails\nB. Compass navigation\nC. Shipbuilding techniques\nD. Port infrastructure": "History",
    "Which has a stronger influence on social behavior: historical narratives or contemporary media? Explain briefly.": "Social Science",
    "Which is more effective in promoting public health: top-down government interventions or grassroots social initiatives? Explain.": "Social Science",
    "Which factor most influences political participation in modern societies?\nA. Economic stability\nB. Education\nC. Media exposure\nD. Social networks": "Social Science"
}

:::{admonition} QUESTION
:class: red
The questions above were generated by ChatGPT. List any concerns that there may be with this approach.
:::

```{admonition} LLM FRAMING: How did I create the questions with ChatGPT?
:class: dropdown, fuchsia
I created these questions with ChatGPT (in the interface) by giving it: 
- Instructions about the original dataset (name of the dataset, on Kaggle, mix of multiple choice Q's, open-ended questions etc.)
- Instructions for what I wanted (history + social science questions, ideally some edge-cases)
- Four examples of questions from the original dataset

I also wrote in the same thread for a long time ... 
- I got ChatGPT to generate a lot of examples, then picked the ones that I liked the most - for the purpose of this class!
- I provided extra instructions such as "give me a few longer questions"

Synthetic data creation is not typically as trivial as my experimenting! This is a huge field on the rise with many considerations on how to do it best.
```

## 1.2 Setup: Install Additional Packages
Please download the packages below (in `venv` or in UCloud) in your terminal:
```bash
pip install sentence-transformers datasets transformers setfit
```

:::{admonition} Download in Jupyter
:class: tip
Remember, you can also download the packages in Jupyter with the `%pip` magic command as we have done in previous classes.
:::

I'll re-import everything (since you worked in a script last), but as always, import only what you need

In [37]:
from pathlib import Path
from datasets import load_dataset, ClassLabel, Dataset # note I have added Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from setfit import SetFitModel, SetFitTrainer

## 1.3 Prepare Data

We have annotated data, but let's prepare it with our previous ds! We'll take 

In [38]:
path = Path.cwd()
data_path = path.parents[1] / "resources" / "data" / "hf" # path for huggingface datasets (so we don't have to redownload them every time)
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Let's downsample to match our 16 history/social science examples. Since we have 4 classes in the original dataset, we'll do 4x8 for our training data. I have cheated got ChatGPT to generate 64 test examples (labelled) for the history/social science examples. So to match this, we'll do 32*4 for the test size!

In [39]:
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))
ds_downsampled = ds.train_test_split(train_size=32,test_size=128, seed=42, stratify_by_column="label")

train_ds = ds_downsampled["train"]
test_ds = ds_downsampled["test"]

Let's combine with our new ds (history and social science)

In [40]:
new_train_data = {
    "question": list(questions.keys()),  # list of questions
    "label": list(questions.values())    # corresponding labels
}

In [41]:
new_train_ds = Dataset.from_dict(new_train_data)

In [42]:
# load val dataset 


## Few-Shot Learning with SetFit 
We'll use the package `setfit`, developed by HuggingFace, to do *few-shot* learning, a specialized type of ML that relies on limited data!

Let's prepare the data firdt

In [43]:
model = SetFitModel.from_pretrained(
    "nomic-ai/modernbert-embed-base", trust_remote_code=True,
)

config.json: 0.00B [00:00, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


### Data

In [44]:
path = Path.cwd()
data_path = path.parents[1] / "resources" / "data" / "hf" # path for huggingface datasets (so we don't have to redownload them every time)

In [45]:
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Print the dataset + an example of a text:

In [46]:
print(ds)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 117519
})


In [47]:
# let's print an example
print(ds["text"][12])

Hydroponic is a subset of what type of culture?
A. Hydroculture
B. Solid medium culture
c. xeroculture
D. Tissue culture


#### Label Column
Let's define the label column:

In [48]:
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))

#### Splitting into Train and Val

We'll split into train and val and downsample to 2000 train examples and 500 test examples to make it run faster for today's class:

In [49]:
# split into train/val
ds = ds.train_test_split(train_size=2000,test_size=500, seed=42, stratify_by_column="label")
train_data = ds["train"]
val_data = ds["test"]

## 1.2 Loading the Model
We'll load the BERT model `distilbert-base-cased` and its corresponding tokenizer. The suffix `cased` tells us that this `DistilBERT` is sensitive to letter case (distinguishing between `english` and `English`).

`DistilBERT` also exists in [uncased](https://huggingface.co/distilbert/distilbert-base-uncased) and [multilingual](https://huggingface.co/distilbert/distilbert-base-multilingual-cased) versions. 

In [50]:
# define model + where to load it from (if already downloaded/cached)
model_path = path.parents[1] / "resources" / "models" / "hf"
model_id = "distilbert/distilbert-base-cased"

# GPU or CPU? Default to CPU if no GPU available


# load model + tokenizer
model = AutoModelForSequenceClassification.from_pretrained(
                                                            model_id, 
                                                            num_labels=num_classes, # pre-defined number of labels in our dataset
                                                            cache_dir=model_path, 
                                                           ) 
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


:::{admonition} "Some weights of DistilBertForSequenceClassification ..." ? 
:class: tip, dropdown
The message above means we *have* to fine-tune the model, since its classification head (weights) is newly initialized. On huggingface.co, you can also find BERT models that have already been fine-tuned for classification.
:::

Let's look more into our model by printing its parameters:

In [51]:
print(model.num_parameters())

65784580


:::{admonition} QUESTION
:class: red
`DistilBERT` has 65.M parameters. From what you might have heard about `Large Language Models` - do you know where this would range? Is this a lot?
:::

## 1.3 Tokenization
We can use `DistilBERT`'s trained tokenizer to represent the text in our dataset:

In [52]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

:::{admonition} QUESTION
:class: red
Can you identify a way to make the function above better in terms of how it is defined and how it is described? Is it easily applicable to other datasets? Why/Why not?

<details>
  <summary>ANSWER</summary>
  I would consider to ...
  <ol>
    <li>Rename the function to <code>tokenize</code>, making its name more informative to its purpose.</li>
    <li>Add more details in the docstring (and <a href="https://docs.python.org/3/library/typing.html">type hints</a>) about the expected input and output, instead of only writing <code>"""Tokenize input data"""</code>.</li>
    <li>Make the function more generalizable by adding a <code>text_col</code> parameter, allowing the user to specify a different column name (e.g., our text column being called <code>"generation"</code>)</li>
  </ol>
</details>
:::

We use the `.map` method to use the tokenize function on each row in our dataset!

In [53]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)

In [54]:
batch_size = 8
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=batch_size,
   per_device_eval_batch_size=batch_size,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

#### Your Turn: Re-create the Pipeline in a Python Script
:::{admonition} HANDS-ON
:class: red
This task focuses on practicing how to create pipelines in scripts. You should do the following: 
1. Draw the DistilBERT fine-tuning pipeline as a diagram on a piece of paper (or digitally in powerpoint). What are the different steps when fine-tuning?
2. Take the snippets above and write them in a Python script
3. Run the script!


**You may structure the script however you like!** You can choose to keep most snippets inside of `main()` or define additional helper functions outside of `main()`, calling them inside of `main()`. See also [Python Scripts](../python_scripts.md).   

For the more advanced coder, try to make the script as generalizable as possible to other BERT models or datasets (e.g., through [argparse](https://docs.python.org/3/howto/argparse.html#introducing-optional-arguments)).
:::